# Variable Registry Audit -- Avenue 1 vs Avenue 2

One row per candidate environmental variable across BOTH avenues: is it ruled out or used, in which avenue, with what evidence, which named experiment/scope it belongs to (a variable can be in more than one), and what it is multicollinear with.

Everything below is computed live from the real code and data -- nothing is a hardcoded snapshot. If a feature set changes in `models/common/torch_data.py` or `models/growth_curve_attribution/broad_environmental_check.py`, re-running this notebook picks that up automatically.

In [1]:
import sys
from pathlib import Path

import pandas as pd

notebook_directory = Path.cwd().resolve()
project_root = next(
    folder for folder in [notebook_directory, *notebook_directory.parents]
    if (folder / "README.md").exists() and (folder / "data").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

# Avenue 1's real feature-set definitions -- imported, not retyped.
from models.common.torch_data import ENV_TERRAIN_FEATURE_SETS
from models.xgb_environmental.xgb_environmental import ALL_FEATURE_COLUMNS
from models.xgb_environmental.grouped_analysis import CATEGORY_GROUPS as AV1_CATEGORY_GROUPS
from models.xgb_environmental.data import load_plots_for_cohort

# Avenue 2's real feature-set definitions -- imported, not retyped.
from models.growth_curve_attribution.broad_environmental_check import FEATURE_GROUPS as AV2_FEATURE_GROUPS
from models.growth_curve_attribution.broad_environmental_check import SCOPE_GROUPS as AV2_SCOPE_GROUPS

print(f"Avenue 1: {len(ALL_FEATURE_COLUMNS)} candidate columns, {len(ENV_TERRAIN_FEATURE_SETS)} named DNN/PINN feature-set tiers")
print(f"Avenue 2: {sum(len(cols) for cols in AV2_FEATURE_GROUPS.values())} candidate columns across {len(AV2_FEATURE_GROUPS)} groups, {len(AV2_SCOPE_GROUPS)} named scopes")

Avenue 1: 48 candidate columns, 5 named DNN/PINN feature-set tiers
Avenue 2: 38 candidate columns across 5 groups, 7 named scopes


## 1. Load the master cross-avenue registry

`documentation/variable_registry_av1_av2.csv` is the existing, already-built record of every variable's status in both avenues. This notebook adds three things that CSV doesn't have: which specific named tier/scope each variable belongs to (not just avenue candidacy), and a live-measured multicollinearity check.

In [2]:
registry = pd.read_csv(project_root / "documentation" / "variable_registry_av1_av2.csv")

# era5_land_temp_k is excluded before it even reaches candidacy in either avenue (only 8 distinct
# values across 71,766 plots -- too coarse), so it never appears in the registry CSV at all.
# Added here by hand as the one genuine exception, so the audit below is complete.
era5_row = pd.DataFrame([{
    "variable": "era5_land_temp_k",
    "provenance": "external (ERA5-Land reanalysis, mean temperature) -- only 8 distinct values across all 71,766 plots, ~11.1km native resolution, far coarser than every other climate source used.",
    "avenue_1_status": "excluded before candidacy -- excluded on the numbers alone, not a correlation/redundancy call",
    "avenue_2_static_status": "never a candidate (Avenue 2 reads the same environmental export Avenue 1 already excluded this from)",
    "avenue_2_temporal_status": "not part of AV2 temporal extension",
    "notes": "the one variable excluded on resolution grounds alone, before any correlation screening could even apply",
}])
registry = pd.concat([registry, era5_row], ignore_index=True)

print(f"{len(registry)} variables in the registry")
registry.head(3)

63 variables in the registry


,variable,provenance,avenue_1_status,avenue_2_static_status,avenue_2_temporal_status,notes
0,CanopyCover,"external (raw survey field, mean canopy cover fraction across every survey y...",used in AV1 candidate universe,used in AV2 management extension,not part of AV2 temporal extension,NaN
1,Thin,"external (raw survey field, fraction of surveys where this plot had been thi...",used in AV1 candidate universe,used in AV2 management extension,not part of AV2 temporal extension,NaN
2,ceh_pedotope,"external (CEH natural capital dataset, natural_capital_pedotopes.tif, one of...",used in AV1 candidate universe,used in AV2 broad-environment extension (soil/site),not part of AV2 temporal extension,categorical; one-hot encoded in AV2 broad-environment checks


## 2. Which named tier/scope each variable actually belongs to

A variable can be in more than one tier -- e.g. `elevation` is in every Avenue 1 DNN/PINN feature set from `terrain_wind_solid` all the way up to `broad_legitimate`. Built by checking real membership in the imported dicts above, not retyped by hand.

In [3]:
def tiers_containing(variable, tier_dict):
    """Return the names of every tier/group/scope in tier_dict that contains this variable."""
    matching_tier_names = []
    for tier_name, tier_columns in tier_dict.items():
        if variable in tier_columns:
            matching_tier_names.append(tier_name)
    return matching_tier_names

registry["av1_dnn_pinn_tiers"] = registry["variable"].apply(lambda v: tiers_containing(v, ENV_TERRAIN_FEATURE_SETS))
registry["av1_category"] = registry["variable"].apply(lambda v: tiers_containing(v, AV1_CATEGORY_GROUPS))
registry["av2_feature_groups"] = registry["variable"].apply(lambda v: tiers_containing(v, AV2_FEATURE_GROUPS))

# Which named AV2 SCOPE (terrain_wind, broad_environment, ...) each variable's own feature GROUP
# feeds into -- a scope is built from one or more groups (see SCOPE_GROUPS), so this expands
# group membership out to the actual experiment names used in the AV2 notebook.
def scopes_containing(feature_groups_for_this_variable):
    matching_scope_names = []
    for scope_name, groups_in_this_scope in AV2_SCOPE_GROUPS.items():
        if any(group in groups_in_this_scope for group in feature_groups_for_this_variable):
            matching_scope_names.append(scope_name)
    return matching_scope_names

registry["av2_scopes"] = registry["av2_feature_groups"].apply(scopes_containing)

registry[["variable", "av1_dnn_pinn_tiers", "av1_category", "av2_feature_groups", "av2_scopes"]].head(10)

,variable,av1_dnn_pinn_tiers,av1_category,av2_feature_groups,av2_scopes
0,CanopyCover,[],[stand_structure],[management],"[terrain_wind_plus_management, broad_environment_plus_management]"
1,Thin,[],[stand_structure],[management],"[terrain_wind_plus_management, broad_environment_plus_management]"
2,ceh_pedotope,[],[soil_site],[soil_site],"[terrain_wind_plus_soil_site, broad_environment, broad_environment_plus_mana..."
3,ceh_subsurface_drainage,[],[soil_site],[soil_site],"[terrain_wind_plus_soil_site, broad_environment, broad_environment_plus_mana..."
4,ceh_textural_composition,[],[soil_site],[soil_site],"[terrain_wind_plus_soil_site, broad_environment, broad_environment_plus_mana..."
5,ceh_twi,"[terrain_wind_solid, terrain_wind_extended, broad, terrain_wind_full, broad_...",[terrain],[terrain_wind],"[terrain_wind, terrain_wind_plus_climate, terrain_wind_plus_soil_site, terra..."
6,chelsa_bio12_precip_mm,[broad_legitimate],[climate],[climate],"[terrain_wind_plus_climate, broad_environment, broad_environment_plus_manage..."
7,chelsa_bio1_celsius,"[broad, broad_legitimate]",[climate],[climate],"[terrain_wind_plus_climate, broad_environment, broad_environment_plus_manage..."
8,chelsa_gdd5_degc,[broad_legitimate],[climate],[climate],"[terrain_wind_plus_climate, broad_environment, broad_environment_plus_manage..."
9,cpmt_compactness_ratio,[broad_legitimate],[spatial_position_edge_effects],[edge_position],"[terrain_wind_plus_edge_position, broad_environment, broad_environment_plus_..."


## 3. Live multicollinearity check

Computed fresh from the real plot data (Spearman correlation, same convention as `grouped_category_importance.ipynb`), not copied from a past run. For each variable, this finds every OTHER candidate variable it correlates with at |rho| >= 0.5 -- both within its own Avenue 1 category and across categories.

In [4]:
plots_df = load_plots_for_cohort("4survey")

# Only correlate columns that are actually numeric and present in the environmental export --
# categorical class IDs (ceh_pedotope etc.) and management/stand-structure columns aren't
# meaningful to Spearman-correlate against terrain/wind/climate/soil columns here.
categorical_columns = ["ceh_pedotope", "ceh_subsurface_drainage", "ceh_textural_composition"]
candidate_numeric_columns = [
    column for column in ALL_FEATURE_COLUMNS
    if column in plots_df.columns and column not in categorical_columns
]

full_correlation_matrix = plots_df[candidate_numeric_columns].corr(method="spearman")
print(f"Correlation matrix computed across {len(candidate_numeric_columns)} numeric candidate columns")

Correlation matrix computed across 45 numeric candidate columns


In [5]:
MULTICOLLINEARITY_THRESHOLD = 0.5

def describe_correlated_partners(variable):
    """Return a short, human-readable string listing every other candidate variable this one
    correlates with above the threshold, sorted by strength -- or an empty string if none."""
    if variable not in full_correlation_matrix.columns:
        return ""
    correlations_with_this_variable = full_correlation_matrix[variable].drop(index=variable)
    strong_partners = correlations_with_this_variable[correlations_with_this_variable.abs() >= MULTICOLLINEARITY_THRESHOLD]
    strong_partners = strong_partners.reindex(strong_partners.abs().sort_values(ascending=False).index)
    if len(strong_partners) == 0:
        return ""
    return "; ".join(f"{partner_name} (rho={value:+.3f})" for partner_name, value in strong_partners.items())

registry["multicollinear_with"] = registry["variable"].apply(describe_correlated_partners)

# A few known-important pairs need a manual note added on top of the raw number, because the
# correlation alone doesn't tell you WHICH one was kept/dropped as a result -- see the two
# 2026-08-06 fixes in documentation/experiment_log.md for the full reasoning.
MULTICOLLINEARITY_RESOLUTION_NOTES = {
    "tpi_250m": " -- REMOVED from Avenue 1 2026-08-06, redundant with both tpi and tpi_500m (Avenue 2's correlation_screen.py found this first)",
    "gwa_wind_speed_50m": " -- Avenue 1's preferred wind representation as of 2026-08-06 (matches Avenue 2's resolved choice)",
    "gwa_wind_speed_10m": " -- superseded by gwa_wind_speed_50m in Avenue 1's DNN/PINN feature sets 2026-08-06 (10m is sub-canopy, backwards for a mature-canopy target)",
    "inverse_slope_proxy": " -- exact duplicate, excluded from Avenue 1's primary terrain_wind_solid feature set",
}
registry["multicollinear_with"] = registry.apply(
    lambda row: row["multicollinear_with"] + MULTICOLLINEARITY_RESOLUTION_NOTES.get(row["variable"], ""),
    axis=1,
)

n_with_partners = (registry["multicollinear_with"] != "").sum()
print(f"{n_with_partners} of {len(registry)} variables have at least one partner at |rho| >= {MULTICOLLINEARITY_THRESHOLD}")

40 of 63 variables have at least one partner at |rho| >= 0.5


## 4. Full merged table

Every column together: provenance, both avenues' status, which named tier/scope it's actually used in, and what it's multicollinear with.

In [6]:
display_columns = [
    "variable", "avenue_1_status", "av1_dnn_pinn_tiers", "av1_category",
    "avenue_2_static_status", "av2_scopes", "multicollinear_with", "notes",
]
full_table = registry[display_columns].sort_values("variable").reset_index(drop=True)
full_table

,variable,avenue_1_status,av1_dnn_pinn_tiers,av1_category,avenue_2_static_status,av2_scopes,multicollinear_with,notes
0,CanopyCover,used in AV1 candidate universe,[],[stand_structure],used in AV2 management extension,"[terrain_wind_plus_management, broad_environment_plus_management]",,NaN
1,Thin,used in AV1 candidate universe,[],[stand_structure],used in AV2 management extension,"[terrain_wind_plus_management, broad_environment_plus_management]",time_since_thinning_missing (rho=-1.000); time_since_thinning (rho=+0.917); ...,NaN
2,ceh_pedotope,used in AV1 candidate universe,[],[soil_site],used in AV2 broad-environment extension (soil/site),"[terrain_wind_plus_soil_site, broad_environment, broad_environment_plus_mana...",,categorical; one-hot encoded in AV2 broad-environment checks
3,ceh_subsurface_drainage,used in AV1 candidate universe,[],[soil_site],used in AV2 broad-environment extension (soil/site),"[terrain_wind_plus_soil_site, broad_environment, broad_environment_plus_mana...",,categorical; one-hot encoded in AV2 broad-environment checks
4,ceh_textural_composition,used in AV1 candidate universe,[],[soil_site],used in AV2 broad-environment extension (soil/site),"[terrain_wind_plus_soil_site, broad_environment, broad_environment_plus_mana...",,categorical; one-hot encoded in AV2 broad-environment checks
...,...,...,...,...,...,...,...,...
58,tpi,used in AV1 candidate universe,"[terrain_wind_full, broad_legitimate]",[terrain],used in AV2 final terrain/wind set,"[terrain_wind, terrain_wind_plus_climate, terrain_wind_plus_soil_site, terra...",profile_curvature (rho=+0.821); ceh_twi (rho=-0.743); plan_curvature (rho=+0...,derived variable
59,tpi_250m,used in AV1 candidate universe,[],[],excluded from AV2 final set after representation check favoured native TPI +...,[],"-- REMOVED from Avenue 1 2026-08-06, redundant with both tpi and tpi_500m (...",derived variable
60,tpi_500m,used in AV1 candidate universe,[],[terrain],used in AV2 final terrain/wind set,"[terrain_wind, terrain_wind_plus_climate, terrain_wind_plus_soil_site, terra...",topex (rho=-0.777); tpi (rho=+0.619); ceh_twi (rho=-0.596); profile_curvatur...,derived variable
61,whcl,used in AV1 candidate universe,"[terrain_wind_extended, broad, terrain_wind_full, broad_legitimate]",[wind],used in AV2 final terrain/wind set,"[terrain_wind, terrain_wind_plus_climate, terrain_wind_plus_soil_site, terra...",,NaN


## 5. Ruled out -- both avenues, with the evidence

Filters to variables NOT in either avenue's final/primary set, so the exclusion reasoning is easy to scan on its own.

In [7]:
ruled_out_mask = (
    registry["avenue_1_status"].str.contains("excluded", case=False, na=False)
    | registry["avenue_2_static_status"].str.contains("excluded|not used|never", case=False, na=False, regex=True)
)
ruled_out_table = registry.loc[ruled_out_mask, ["variable", "avenue_1_status", "avenue_2_static_status", "multicollinear_with"]].sort_values("variable")
print(f"{len(ruled_out_table)} variables ruled out somewhere")
ruled_out_table

24 variables ruled out somewhere


,variable,avenue_1_status,avenue_2_static_status,multicollinear_with
62,era5_land_temp_k,"excluded before candidacy -- excluded on the numbers alone, not a correlatio...",never a candidate (Avenue 2 reads the same environmental export Avenue 1 alr...,
21,gwa_prob_above_critical_10m,used in AV1 candidate universe,excluded from AV2 final set: derived / redundant alternative wind representa...,gwa_wind_p95_10m (rho=+0.916); gwa_weibull_a_10m (rho=+0.836); gwa_weibull_k...
22,gwa_prob_above_critical_50m,used in AV1 candidate universe,excluded from AV2 final set: redundant alternative wind representation,gwa_wind_p95_50m (rho=+0.976); gwa_wind_speed_50m (rho=+0.911); gwa_weibull_...
23,gwa_weibull_a_10m,used in AV1 candidate universe,excluded from AV2 final set: redundant alternative wind representation,gwa_wind_p95_10m (rho=+0.983); gwa_prob_above_critical_10m (rho=+0.836); gwa...
24,gwa_weibull_a_50m,used in AV1 candidate universe,excluded from AV2 final set: redundant alternative wind representation,gwa_wind_speed_50m (rho=+1.000); gwa_wind_p95_50m (rho=+0.972); gwa_prob_abo...
25,gwa_weibull_k_10m,used in AV1 candidate universe,excluded from AV2 final set: redundant alternative wind representation,gwa_prob_above_critical_10m (rho=-0.608)
26,gwa_weibull_k_50m,used in AV1 candidate universe,excluded from AV2 final set: redundant alternative wind representation,chelsa_bio12_precip_mm (rho=+0.587)
27,gwa_wind_p95_10m,used in AV1 candidate universe,excluded from AV2 final set: derived / redundant alternative wind representa...,gwa_weibull_a_10m (rho=+0.983); gwa_prob_above_critical_10m (rho=+0.916); gw...
28,gwa_wind_p95_50m,used in AV1 candidate universe,excluded from AV2 final set: redundant alternative wind representation,gwa_wind_speed_50m (rho=+0.978); gwa_prob_above_critical_50m (rho=+0.976); g...
29,gwa_wind_speed_10m,used in AV1 candidate universe,excluded from AV2 final set after representation swap to 50m wind,gwa_weibull_a_10m (rho=+0.678); gwa_wind_p95_10m (rho=+0.672); gwa_prob_abov...


## 6. Used variables, grouped by named experiment

One row per (variable, tier/scope) pair -- since a variable can appear in several, this is a long-format view that's easy to filter down to just one experiment at a time (e.g. `av1_broad_legitimate_rows` below).

In [8]:
long_format_rows = []
for _, row in registry.iterrows():
    for tier_name in row["av1_dnn_pinn_tiers"]:
        long_format_rows.append({"variable": row["variable"], "avenue": "AV1", "experiment": tier_name})
    for scope_name in row["av2_scopes"]:
        long_format_rows.append({"variable": row["variable"], "avenue": "AV2", "experiment": scope_name})

long_format_table = pd.DataFrame(long_format_rows)

print("Variable count per named experiment:")
print(long_format_table.groupby(["avenue", "experiment"])["variable"].count().sort_index())

# Example of filtering to one experiment -- edit the two lines below to inspect any tier/scope.
av1_broad_legitimate_rows = long_format_table[long_format_table["experiment"] == "broad_legitimate"]
av1_broad_legitimate_rows

Variable count per named experiment:
avenue  experiment                       
AV1     broad                                10
        broad_legitimate                     27
        terrain_wind_extended                 7
        terrain_wind_full                    16
        terrain_wind_solid                    5
AV2     broad_environment                    33
        broad_environment_plus_management    38
        terrain_wind                         17
        terrain_wind_plus_climate            22
        terrain_wind_plus_edge_position      23
        terrain_wind_plus_management         22
        terrain_wind_plus_soil_site          22
Name: variable, dtype: int64


,variable,avenue,experiment
17,ceh_twi,AV1,broad_legitimate
25,chelsa_bio12_precip_mm,AV1,broad_legitimate
30,chelsa_bio1_celsius,AV1,broad_legitimate
34,chelsa_gdd5_degc,AV1,broad_legitimate
38,cpmt_compactness_ratio,AV1,broad_legitimate
42,dist_to_block_boundary,AV1,broad_legitimate
46,dist_to_cpmt_boundary,AV1,broad_legitimate
50,dist_to_forest_perimeter,AV1,broad_legitimate
54,dist_to_road,AV1,broad_legitimate
58,dist_to_scpt_boundary,AV1,broad_legitimate


## Key findings from this audit

1. **`tpi_250m`** was genuinely redundant (rho=0.841 with `tpi`, rho=0.879 with `tpi_500m`) -- Avenue 2 found this first via `correlation_screen.py`; Avenue 1 had it in its candidate set until fixed 2026-08-06.
2. **The GWA Weibull family is severely multicollinear, now measured directly rather than just flagged**: `gwa_wind_speed_50m`/`gwa_weibull_a_50m` correlate at rho=+1.000 (perfectly redundant), and the whole 50m cluster (wind_speed, weibull_a, wind_p95, prob_above_critical) sits at rho=0.90-1.00 with itself -- essentially one variable's worth of information represented four ways. The 10m cluster shows the same pattern (rho=0.68-0.98). This was previously only a caveat in the provenance text; it had never actually been measured until this notebook.
3. **`slope_degrees`/`inverse_slope_proxy`** are an exact duplicate (rho=-1.000) -- both avenues exclude the same one.
4. **Management means the same 5 columns in both avenues** (`CanopyCover`, `Thin`, `time_since_thinning`, `time_since_thinning_missing`, `recent_thinning_5yr`) -- confirmed identical, not just similarly-named. The difference is architectural: Avenue 1 feeds them to every model by default (never an optional lever), Avenue 2 treats them as one of several scopes to switch on/off.
5. **`gwa_wind_speed_10m` vs `50m`**: Avenue 2 resolved this (10m is sub-canopy, backwards for a mature-canopy target) before Avenue 1 did -- fixed 2026-08-06 to match.

See `documentation/experiment_log.md`'s 2026-08-06 entries for the full reasoning chain behind each fix.